# Time Series Analysis Dashboard

This notebook provides comprehensive time series analysis and forecasting capabilities for agricultural data.

In [ ]:
# Import Required Libraries for Time Series Analysis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Time series specific imports
try:
    from statsmodels.tsa.seasonal import seasonal_decompose
    from statsmodels.tsa.arima.model import ARIMA
    from statsmodels.tsa.stattools import adfuller
    from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
    print("Time series libraries imported successfully!")
except ImportError as e:
    print(f"Warning: Some time series libraries not available: {e}")
    print("You may need to install: pip install statsmodels")

# Set style
plt.style.use('default')
sns.set_palette("viridis")

print("All libraries imported successfully!")

## Load Time Series Dataset

In [ ]:
# Load time series dataset
ts_data = pd.read_csv('recommended_time_series_dataset.csv')

print("Dataset Shape:", ts_data.shape)
print("\nColumn Names:", ts_data.columns.tolist())
print("\nFirst 5 rows:")
print(ts_data.head())

print("\nDataset Info:")
print(ts_data.info())

print("\nDate range:")
print(f"Start: {ts_data['Date'].min()}")
print(f"End: {ts_data['Date'].max()}")

print("\nUnique Crop Types:")
print(ts_data['Crop_Type'].unique())

print("\nBasic Statistics:")
print(ts_data.describe())

In [ ]:
# Load time series dataset
ts_data = pd.read_csv('first_fast_api_project/Time_Series_Model/recommended_time_series_dataset.csv')

print("Dataset Shape:", ts_data.shape)
print("\nColumn Names:", ts_data.columns.tolist())
print("\nFirst 5 rows:")
print(ts_data.head())

print("\nDataset Info:")
print(ts_data.info())

print("\nDate range:")
print(f"From: {ts_data['Date'].min()}")
print(f"To: {ts_data['Date'].max()}")

print("\nUnique crop types:")
print(ts_data['Crop_Type'].unique())

print("\nBasic Statistics:")
print(ts_data.describe())

## Time Series Data Preprocessing

In [ ]:
# Convert Date column to datetime
ts_data['Date'] = pd.to_datetime(ts_data['Date'])

# Sort by date
ts_data = ts_data.sort_values('Date')

# Set Date as index for time series analysis
ts_data.set_index('Date', inplace=True)

print("Data after preprocessing:")
print(f"Shape: {ts_data.shape}")
print(f"Date range: {ts_data.index.min()} to {ts_data.index.max()}")
print(f"Missing values: {ts_data.isnull().sum().sum()}")

# Create monthly aggregated data for better analysis
monthly_data = ts_data.resample('M').agg({
    'Crop_Yield': 'mean',
    'Temperature': 'mean',
    'Humidity': 'mean',
    'Wind_Speed': 'mean',
    'Soil_Quality': 'mean'
}).round(2)

print(f"\nMonthly aggregated data shape: {monthly_data.shape}")
print("\nMonthly data sample:")
print(monthly_data.head())

# Create crop-specific time series
crop_yields = {}
for crop in ts_data['Crop_Type'].unique():
    crop_data = ts_data[ts_data['Crop_Type'] == crop]['Crop_Yield']
    crop_monthly = crop_data.resample('M').mean()
    crop_yields[crop] = crop_monthly.dropna()
    print(f"{crop}: {len(crop_monthly)} monthly data points")

print(f"\nProcessing complete. Ready for time series analysis!")

## Time Series Analysis and Forecasting

In [ ]:
# Create comprehensive time series visualization dashboard
fig = make_subplots(
    rows=4, cols=2,
    subplot_titles=[
        'Overall Crop Yield Trend', 'Yield by Crop Type Over Time',
        'Monthly Yield Distribution', 'Temperature vs Yield Correlation',
        'Seasonal Decomposition - Trend', 'Seasonal Decomposition - Seasonal',
        'Yield Forecasting', 'Environmental Factors Over Time'
    ],
    specs=[[{"type": "scatter"}, {"type": "scatter"}],
           [{"type": "box"}, {"type": "scatter"}],
           [{"type": "scatter"}, {"type": "scatter"}],
           [{"type": "scatter"}, {"type": "scatter"}]]
)

# 1. Overall crop yield trend
fig.add_trace(
    go.Scatter(
        x=monthly_data.index,
        y=monthly_data['Crop_Yield'],
        mode='lines+markers',
        name='Overall Yield Trend',
        line=dict(color='blue', width=2)
    ),
    row=1, col=1
)

# 2. Yield by crop type over time
colors = ['red', 'green', 'purple', 'orange', 'brown', 'pink', 'gray', 'olive']
for i, (crop, yield_data) in enumerate(crop_yields.items()):
    if len(yield_data) > 0:  # Only plot if data exists
        fig.add_trace(
            go.Scatter(
                x=yield_data.index,
                y=yield_data.values,
                mode='lines',
                name=crop,
                line=dict(color=colors[i % len(colors)]),
                showlegend=False
            ),
            row=1, col=2
        )

# 3. Monthly yield distribution (box plot by month)
monthly_data_reset = monthly_data.reset_index()
monthly_data_reset['Month'] = monthly_data_reset['Date'].dt.month
for month in range(1, 13):
    month_data = monthly_data_reset[monthly_data_reset['Month'] == month]['Crop_Yield']
    if len(month_data) > 0:
        fig.add_trace(
            go.Box(
                y=month_data,
                name=f'Month {month}',
                showlegend=False
            ),
            row=2, col=1
        )

# 4. Temperature vs Yield correlation
fig.add_trace(
    go.Scatter(
        x=monthly_data['Temperature'],
        y=monthly_data['Crop_Yield'],
        mode='markers',
        name='Temp vs Yield',
        marker=dict(color='red', opacity=0.6),
        showlegend=False
    ),
    row=2, col=2
)

# 5 & 6. Seasonal decomposition (if statsmodels is available)
try:
    decomposition = seasonal_decompose(monthly_data['Crop_Yield'].dropna(), model='additive', period=12)
    
    # Trend
    fig.add_trace(
        go.Scatter(
            x=decomposition.trend.index,
            y=decomposition.trend.values,
            mode='lines',
            name='Trend',
            line=dict(color='green'),
            showlegend=False
        ),
        row=3, col=1
    )
    
    # Seasonal
    fig.add_trace(
        go.Scatter(
            x=decomposition.seasonal.index,
            y=decomposition.seasonal.values,
            mode='lines',
            name='Seasonal',
            line=dict(color='orange'),
            showlegend=False
        ),
        row=3, col=2
    )
except:
    print("Seasonal decomposition not available - need statsmodels")

# 7. Simple forecasting (linear trend)
recent_data = monthly_data['Crop_Yield'].dropna().tail(12)
if len(recent_data) > 0:
    # Simple linear forecast
    x_vals = np.arange(len(recent_data))
    z = np.polyfit(x_vals, recent_data.values, 1)
    trend_line = np.poly1d(z)
    
    # Forecast next 6 months
    future_x = np.arange(len(recent_data), len(recent_data) + 6)
    forecast = trend_line(future_x)
    
    future_dates = pd.date_range(start=recent_data.index[-1], periods=7, freq='M')[1:]
    
    # Historical
    fig.add_trace(
        go.Scatter(
            x=recent_data.index,
            y=recent_data.values,
            mode='lines+markers',
            name='Historical',
            line=dict(color='blue'),
            showlegend=False
        ),
        row=4, col=1
    )
    
    # Forecast
    fig.add_trace(
        go.Scatter(
            x=future_dates,
            y=forecast,
            mode='lines+markers',
            name='Forecast',
            line=dict(color='red', dash='dash'),
            showlegend=False
        ),
        row=4, col=1
    )

# 8. Environmental factors over time
fig.add_trace(
    go.Scatter(
        x=monthly_data.index,
        y=monthly_data['Temperature'],
        mode='lines',
        name='Temperature',
        line=dict(color='red'),
        showlegend=False
    ),
    row=4, col=2
)

fig.add_trace(
    go.Scatter(
        x=monthly_data.index,
        y=monthly_data['Humidity'],
        mode='lines',
        name='Humidity',
        line=dict(color='blue'),
        yaxis='y2',
        showlegend=False
    ),
    row=4, col=2
)

# Update layout
fig.update_layout(
    height=1600,
    title_text="Comprehensive Time Series Analysis Dashboard",
    title_x=0.5,
    showlegend=False
)

fig.show()